In [ ]:
# =========================================
# IMPORT LIBRARIES
# =========================================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression

from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

print("Libraries imported successfully!")

In [ ]:
# =========================================
# COLUMN NAMES
# =========================================

column_names = [
    'age',
    'sex',
    'cp',
    'trestbps',
    'chol',
    'fbs',
    'restecg',
    'thalach',
    'exang',
    'oldpeak',
    'slope',
    'ca',
    'thal',
    'num'
]

# =========================================
# LOAD DATASET
# =========================================

df = pd.read_csv(
    "Cleavland.csv",
    names=column_names
)

print("FIRST 5 ROWS")
print(df.head())

print("\nDATASET SHAPE")
print(df.shape)


In [ ]:
# =========================================
# (a) DATA CLEANING
# =========================================

# Replace ? with NaN
df.replace("?", np.nan, inplace=True)

# Convert columns to numeric
for col in df.columns:

    df[col] = pd.to_numeric(
        df[col],
        errors='coerce'
    )

# Check missing values
print("\nMISSING VALUES")
print(df.isnull().sum())

# Remove missing values
df_cleaned = df.dropna()

# Remove negative values
numeric_cols = df_cleaned.select_dtypes(include=[np.number]).columns

df_cleaned = df_cleaned[
    (df_cleaned[numeric_cols] >= 0).all(axis=1)
]

print("\nSHAPE AFTER CLEANING")
print(df_cleaned.shape)


In [ ]:
# =========================================
# (b) ERROR CORRECTING
# OUTLIER DETECTION & REMOVAL (IQR)
# =========================================

def remove_outliers_iqr(df, columns):

    for col in columns:

        Q1 = df[col].quantile(0.25)

        Q3 = df[col].quantile(0.75)

        IQR = Q3 - Q1

        lower = Q1 - 1.5 * IQR

        upper = Q3 + 1.5 * IQR

        df = df[
            (df[col] >= lower) &
            (df[col] <= upper)
        ]

    return df

# Columns used for outlier removal
outlier_cols = ['trestbps', 'chol', 'thalach', 'oldpeak']

df_no_outliers = remove_outliers_iqr(
    df_cleaned.copy(),
    outlier_cols
)

print("\nSHAPE AFTER OUTLIER REMOVAL")
print(df_no_outliers.shape)


In [ ]:
# =========================================
# (c) DATA TRANSFORMATION
# FEATURE SCALING
# =========================================

# Features and target
# Binary target: 0 = No Heart Disease, 1 = Heart Disease
X = df_no_outliers.drop('num', axis=1)

y = (df_no_outliers['num'] > 0).astype(int)

cols_to_scale = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X[cols_to_scale])

# Replace scaled columns back into X
X = X.copy()
X[cols_to_scale] = X_scaled

print("\nDATA TRANSFORMATION COMPLETED")
print(X.head())


In [ ]:
# =========================================
# TRAIN TEST SPLIT
# =========================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Train size:", X_train.shape[0], "| Test size:", X_test.shape[0])

In [ ]:
# =========================================
# (d) LOGISTIC REGRESSION MODEL
# =========================================

logreg = LogisticRegression(max_iter=1000)

logreg.fit(X_train, y_train)

logreg_pred = logreg.predict(X_test)

accuracy_lr = accuracy_score(y_test, logreg_pred)

print("Logistic Regression Accuracy:", round(accuracy_lr * 100, 2), "%")


In [ ]:
# =========================================
# KNN MODEL
# =========================================

knn = KNeighborsClassifier(n_neighbors=5)

knn.fit(X_train, y_train)

knn_pred = knn.predict(X_test)

accuracy_knn = accuracy_score(y_test, knn_pred)

print("KNN Accuracy:", round(accuracy_knn * 100, 2), "%")

In [ ]:
# =========================================
# ACCURACY COMPARISON
# =========================================

print("\n================================")
print("MODEL ACCURACY")
print("================================")

print("\nLogistic Regression Accuracy:")
print(accuracy_lr)

print("\nKNN Accuracy:")
print(accuracy_knn)


In [ ]:
# =========================================
# CLASSIFICATION REPORTS
# =========================================

print("\n================================")
print("\nLOGISTIC REGRESSION REPORT")

print(classification_report(
    y_test,
    logreg_pred
))

print("\n================================")
print("KNN REPORT")
print("================================")

print(classification_report(
    y_test,
    knn_pred
))

In [ ]:
# =========================================
# CONFUSION MATRIX VISUALIZATION
# LOGISTIC REGRESSION
# =========================================

sns.heatmap(
    confusion_matrix(y_test, logreg_pred),
    annot=True,
    fmt='d'
)

plt.title("Logistic Regression Confusion Matrix")

plt.show()

# =========================================
# CONFUSION MATRIX VISUALIZATION
# KNN
# =========================================

sns.heatmap(
    confusion_matrix(y_test, knn_pred),
    annot=True,
    fmt='d'
)

plt.title("KNN Confusion Matrix")

plt.show()

# =========================================
# FINAL COMPARISON
# =========================================

print("\n================================")
print("FINAL COMPARISON")
print("================================")

if accuracy_lr > accuracy_knn:

    print("\nLogistic Regression Performs Better")

elif accuracy_knn > accuracy_lr:

    print("\nKNN Performs Better")

else:

    print("\nBoth Models have Equal Accuracy")